# CPTAC External Validation: FiLM-Conditioned Attention-MIL

Runs the 5 TCGA-trained fold checkpoints on held-out CPTAC LUAD/LUSC cases.

**Expected Kaggle datasets attached to this notebook:**
1. `cptac-luad-part1`
2. `cptac-luad-part2`
3. `cptac-lusc`
4. `cptac-nsclc-metadata` — contains `cptac-nsclc-metadata.csv`
5. `nsclc-film-mil-checkpoints` — contains `fold0..4_best_model.pt` **and** `model_config.json`
   (make sure `model_config.json` is in this dataset — it's not listed as a separate
   dataset above, so it needs to live alongside the checkpoints)

This notebook does **not** need `train_film_mil.py`'s `FiLMDataset` — CPTAC file naming
breaks its fixed 12-character barcode slicing (TCGA-specific), and it only supports one
source directory per subtype. A dedicated `CPTACExternalDataset` below fixes both, and
concatenates all of a patient's slides into a single tile bag (one prediction per
patient, using all available tissue, without inflating the test-set N).

## Setup

In [ ]:
!pip install h5py --quiet

import os, json, re, shutil, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

In [ ]:
# Dataset paths
LUAD_DIRS = [
    "/kaggle/input/datasets/lucashuitema/cptac-luad-part1",
    "/kaggle/input/datasets/lucashuitema/cptac-luad-part2",
]
LUSC_DIRS = [
    "/kaggle/input/datasets/lucashuitema/cptac-lusc",
]
METADATA_CSV = "/kaggle/input/datasets/lucashuitema/cptac-csv/cptac-nsclc-metadata.csv"

# One Kaggle dataset per architecture, each containing that architecture's
# fold*.pt checkpoints (as saved by the training notebook's per-architecture
# AGG_OUTPUT_DIR). Comment out / remove any entry you don't have checkpoints
# for -- everything below adapts automatically to however many architectures
# are present here.
CHECKPOINT_DIRS = {
    "abmil":    "/kaggle/input/datasets/lucashuitema/seed96-nofilm-mil-abmil",
    "clam":     "/kaggle/input/datasets/lucashuitema/seed96-nofilm-mil-clam",
    "transmil": "/kaggle/input/datasets/lucashuitema/seed96-nofilm-mil-transmil",
    "graph":    "/kaggle/input/datasets/lucashuitema/seed96-nofilm-mil-graph",
}

# gene_cols/clinical_cols/APM_GENES/TIS_GENES/n_folds are identical across
# architectures (same TCGA training data + split), so one shared config is
# fine -- point this at any one architecture's model_config.json.
MODEL_CONFIG_DIR = "/kaggle/input/datasets/lucashuitema/nsclc-mil-checkpoints-no-film"

OUTPUT_DIR = "/kaggle/working/cptac_external_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILM_ENABLED = False # Set to true or false depending if you trained the model with the FiLM layer enabled
USE_PANEL_HEAD = True

# Aggregator hyperparameters used to reconstruct each architecture's model.
# MUST match whatever was used at training time -- if you changed any of
# these away from train_film_mil.py's defaults when training, update them
# here too, or load_state_dict will fail on shape mismatches (not just
# key-name mismatches).
AGG_HIDDEN_DIM = 256
GRAPH_K = 8
GRAPH_LAYERS = 2
TRANSMIL_HEADS = 8
TRANSMIL_LAYERS = 2
TRANSMIL_DIM_FEEDFORWARD = 512

for p in LUAD_DIRS + LUSC_DIRS + [METADATA_CSV] + list(CHECKPOINT_DIRS.values()):
    exists = Path(p).exists()
    print(f"{'OK ' if exists else 'MISSING'}  {p}")
    if not exists:
        print(f"Fix the missing path(s) before continuing")

## Model definition

Mirrors train_film_mil.py exactly (all 4 aggregator architectures: abmil,
clam, transmil, graph), so no import is required and each architecture's
checkpoint loads with load_state_dict without modification.

In [ ]:
class AttentionMIL(nn.Module):
    """Additive attention pooling (Ilse et al., 2018)."""
    def __init__(self, feat_dim: int = 1536, hidden_dim: int = 256):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.feat_proj = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.25),
        )

    def forward(self, features):
        assert features.ndim == 2, (
            f"AttentionMIL expects (N_tiles, feat_dim), got shape {tuple(features.shape)}"
        )
        projected = self.feat_proj(features)            # (N, 512)
        raw_attn  = self.attention(features)             # (N, 1)
        attn      = torch.softmax(raw_attn, dim=0)        # (N, 1)
        slide_embed = (attn * projected).sum(dim=0)       # (512,)
        return slide_embed, attn.squeeze(-1)


class CLAMGatedAttentionMIL(nn.Module):
    """
    CLAM-style single-branch gated-attention pooling (Lu et al., 2021).
    Attention logits come from a gated V(x) * sigmoid(U(x)) combination
    instead of a single tanh MLP. Reproduces CLAM's attention-pooling
    backbone only (no instance-level clustering loss -- that assumes a
    discrete class label, which doesn't apply to continuous regression).
    """
    def __init__(self, feat_dim: int = 1536, hidden_dim: int = 256,
                 embed_dim: int = 512, dropout: float = 0.25):
        super().__init__()
        self.attention_V = nn.Sequential(nn.Linear(feat_dim, hidden_dim), nn.Tanh())
        self.attention_U = nn.Sequential(nn.Linear(feat_dim, hidden_dim), nn.Sigmoid())
        self.attention_w = nn.Linear(hidden_dim, 1)
        self.feat_proj = nn.Sequential(
            nn.Linear(feat_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, features):
        assert features.ndim == 2, (
            f"CLAMGatedAttentionMIL expects (N_tiles, feat_dim), got shape "
            f"{tuple(features.shape)}"
        )
        projected = self.feat_proj(features)                       # (N, embed_dim)
        gated_logits = self.attention_w(
            self.attention_V(features) * self.attention_U(features)
        )                                                            # (N, 1)
        attn = torch.softmax(gated_logits, dim=0)                   # (N, 1) sums to 1
        slide_embed = (attn * projected).sum(dim=0)                 # (embed_dim,)
        return slide_embed, attn.squeeze(-1)


class TransMILAggregator(nn.Module):
    """
    Compact Transformer MIL aggregator, TransMIL-style (Shao et al., 2021):
    tiles attend to each other via self-attention instead of being pooled
    independently, and a learnable [CLS] token summarises the bag.
    Uses standard self-attention (not Nystrom) and omits PPEG (no 2D tile
    grid consumed elsewhere in this pipeline) -- see train_film_mil.py's
    TransMILAggregator docstring for the full simplification notes.
    """
    def __init__(self, feat_dim: int = 1536, embed_dim: int = 512,
                 n_heads: int = 8, n_layers: int = 2,
                 dim_feedforward: int = 512, dropout: float = 0.25):
        super().__init__()
        self.input_proj = nn.Linear(feat_dim, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, features):
        assert features.ndim == 2, (
            f"TransMILAggregator expects (N_tiles, feat_dim), got shape "
            f"{tuple(features.shape)}"
        )
        tokens = self.input_proj(features)                          # (N, embed_dim)
        tokens = torch.cat([self.cls_token, tokens], dim=0).unsqueeze(0)  # (1, N+1, embed_dim)
        encoded = self.encoder(tokens).squeeze(0)                    # (N+1, embed_dim)
        slide_embed = self.norm(encoded[0])                          # CLS token -> (embed_dim,)
        with torch.no_grad():
            tile_repr = encoded[1:]
            attn_proxy = F.cosine_similarity(
                tile_repr, slide_embed.unsqueeze(0).expand_as(tile_repr), dim=-1
            )
            attn_proxy = torch.softmax(attn_proxy, dim=0)
        return slide_embed, attn_proxy


class GraphMILAggregator(nn.Module):
    """
    Lightweight Graph-MIL aggregator: tiles are nodes of a feature-space
    k-NN graph (cosine similarity), messages pass through residual
    mean-aggregation graph-conv layers, then an attention-pooling readout
    (same family as AttentionMIL/CLAM) produces the slide embedding. Plain
    PyTorch, no torch_geometric dependency.
    """
    def __init__(self, feat_dim: int = 1536, embed_dim: int = 512,
                 hidden_dim: int = 256, k: int = 8, n_layers: int = 2,
                 dropout: float = 0.25):
        super().__init__()
        self.k = k
        self.input_proj = nn.Sequential(nn.Linear(feat_dim, hidden_dim), nn.ReLU())
        self.gnn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
            for _ in range(n_layers)
        ])
        self.out_proj = nn.Linear(hidden_dim, embed_dim)
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1)
        )

    def _knn_indices(self, x: torch.Tensor):
        n = x.shape[0]
        if n <= 1:
            return None
        with torch.no_grad():
            x_norm = F.normalize(x, dim=-1)
            sim = x_norm @ x_norm.t()                 # (N, N)
            sim.fill_diagonal_(-float("inf"))
            k = min(self.k, n - 1)
            _, idx = sim.topk(k, dim=-1)               # (N, k)
        return idx

    def forward(self, features):
        assert features.ndim == 2, (
            f"GraphMILAggregator expects (N_tiles, feat_dim), got shape "
            f"{tuple(features.shape)}"
        )
        h = self.input_proj(features)                  # (N, hidden_dim)
        n = h.shape[0]
        idx = self._knn_indices(h)
        for layer in self.gnn_layers:
            neighbor_mean = h[idx].mean(dim=1) if idx is not None else h  # (N, hidden_dim)
            h = layer(torch.cat([h, neighbor_mean], dim=-1)) + h          # residual message passing
        node_embed = self.out_proj(h)                    # (N, embed_dim)
        attn_logits = self.attention(node_embed)          # (N, 1)
        attn = torch.softmax(attn_logits, dim=0)
        slide_embed = (attn * node_embed).sum(dim=0)      # (embed_dim,)
        return slide_embed, attn.squeeze(-1)


AGGREGATOR_CHOICES = ["abmil", "clam", "transmil", "graph"]


def build_aggregator(name: str, feat_dim: int, embed_dim: int,
                      hidden_dim: int = 256, dropout: float = 0.25,
                      **kwargs) -> nn.Module:
    """Factory matching train_film_mil.py's build_aggregator() exactly --
    same class, same attribute names, so state_dicts saved by the training
    script load here without modification."""
    name = name.lower()
    if name == "abmil":
        return AttentionMIL(feat_dim=feat_dim, hidden_dim=hidden_dim)
    elif name == "clam":
        return CLAMGatedAttentionMIL(
            feat_dim=feat_dim, hidden_dim=hidden_dim, embed_dim=embed_dim, dropout=dropout,
        )
    elif name == "transmil":
        return TransMILAggregator(
            feat_dim=feat_dim,
            embed_dim=embed_dim,
            n_heads=kwargs.get("transmil_heads", 8),
            n_layers=kwargs.get("transmil_layers", 2),
            dim_feedforward=kwargs.get("transmil_dim_feedforward", 512),
            dropout=dropout,
        )
    elif name == "graph":
        return GraphMILAggregator(
            feat_dim=feat_dim,
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            k=kwargs.get("graph_k", 8),
            n_layers=kwargs.get("graph_layers", 2),
            dropout=dropout,
        )
    else:
        raise ValueError(f"Unknown aggregator '{name}'. Choose from: {AGGREGATOR_CHOICES}")


class FiLMLayer(nn.Module):
    """Feature-wise Linear Modulation (Perez et al., 2018)."""
    def __init__(self, embed_dim: int = 512, n_subtypes: int = 2):
        super().__init__()
        self.subtype_embed = nn.Embedding(n_subtypes, embed_dim)
        self.gamma = nn.Linear(embed_dim, embed_dim)
        self.beta  = nn.Linear(embed_dim, embed_dim)

    def forward(self, slide_embed, subtype_id):
        s     = self.subtype_embed(subtype_id)
        gamma = self.gamma(s)
        beta  = self.beta(s)
        return gamma * slide_embed + beta


class FiLMMILModel(nn.Module):
    """End-to-end FiLM-conditioned MIL model. Architecture must match
    train_film_mil.py EXACTLY (including the Sequential blocks in
    feat_proj/head, and which aggregator is selected) or load_state_dict
    will fail on key names/shapes. The aggregator submodule is always kept
    under the historical attribute name `attention_mil` regardless of which
    architecture it actually is -- that's what lets every architecture's
    checkpoint use the same key prefix."""
    def __init__(
        self,
        feat_dim:     int = 1536,
        embed_dim:    int = 512,
        clinical_dim: int = 2,
        n_clinical:   int = 64,
        n_genes:      int = 35,
        n_subtypes:   int = 2,
        dropout:      float = 0.25,
        *,
        use_film:     bool,
        use_panel_head: bool = True,
        aggregator:    str = "abmil",
        agg_hidden_dim: int = 256,
        graph_k:       int = 8,
        graph_layers:  int = 2,
        transmil_heads: int = 8,
        transmil_layers: int = 2,
        transmil_dim_feedforward: int = 512,
    ):
        super().__init__()
        self.use_film = use_film
        self.use_panel_head = use_panel_head
        self.aggregator_name = aggregator
        self.attention_mil = build_aggregator(
            aggregator,
            feat_dim=feat_dim,
            embed_dim=embed_dim,
            hidden_dim=agg_hidden_dim,
            dropout=dropout,
            graph_k=graph_k,
            graph_layers=graph_layers,
            transmil_heads=transmil_heads,
            transmil_layers=transmil_layers,
            transmil_dim_feedforward=transmil_dim_feedforward,
        )
        if self.use_film:
            self.film = FiLMLayer(embed_dim=embed_dim, n_subtypes=n_subtypes)
        self.clinical_proj = nn.Sequential(
            nn.Linear(clinical_dim, n_clinical),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(embed_dim + n_clinical, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, n_genes),
        )
        if self.use_panel_head:
            self.panel_head = nn.Sequential(
                nn.Linear(embed_dim + n_clinical, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, 2),   # [APM, TIS]
            )

    def forward(self, features, clinical, subtype_id):
        slide_embed, attn_weights = self.attention_mil(features)
        if self.use_film:
            slide_embed = self.film(slide_embed, subtype_id)
        clin_embed  = self.clinical_proj(clinical)
        combined = torch.cat([slide_embed, clin_embed], dim=-1)
        preds    = self.head(combined)
        return preds, attn_weights


print("Model classes defined (4 aggregators: abmil, clam, transmil, graph).")

## CPTAC-specific Dataset

Regex-based case ID extraction (handles CPTAC's variable-length slide suffixes, unlike
the training script's fixed 12-char slice), multi-directory support per subtype, and
multi-slide concatenation into one tile bag per patient.

In [ ]:
CASE_ID_RE = re.compile(r"(C3[LN]-\d{5})", re.IGNORECASE)


class CPTACExternalDataset(Dataset):
    def __init__(self, df, feature_dirs, gene_cols, clinical_cols, n_tiles_cap=None):
        """
        feature_dirs: {"LUAD": [dir1, dir2, ...], "LUSC": [dir1, ...]}
        n_tiles_cap: optional cap on total tiles per patient (random subsample), if it gets too large for GPU memory.
        """
        self.gene_cols = gene_cols
        self.clinical_cols = clinical_cols
        self.n_tiles_cap = n_tiles_cap

        self.case_index = {}
        for subtype, dirs in feature_dirs.items():
            index = {}
            for d in dirs:
                for p in Path(d).glob("*.h5"):
                    m = CASE_ID_RE.search(p.stem)
                    if m is None:
                        continue
                    cid = m.group(1).upper()
                    index.setdefault(cid, []).append(p)
            self.case_index[subtype] = index
            n_slides = sum(len(v) for v in index.values())
            print(f"{subtype}: {len(index)} unique cases, {n_slides} total slides across {len(dirs)} source dir(s)")

        records = []
        n_ambiguous = 0
        for _, row in df.iterrows():
            sid = str(row["submitter_id"]).upper().strip()
            found = {st: idx[sid] for st, idx in self.case_index.items() if sid in idx}
            if len(found) == 0:
                continue
            if len(found) > 1:
                n_ambiguous += 1
                continue
            subtype, h5_paths = next(iter(found.items()))
            records.append({
                "sid": sid,
                "subtype": subtype,
                "h5_paths": h5_paths,
                "n_slides": len(h5_paths),
                "label": row[gene_cols].values.astype(np.float32),
                "clinical": row[clinical_cols].values.astype(np.float32),
            })
        self.records = records
        if n_ambiguous:
            print(f"\u26a0 skipped {n_ambiguous} case(s) found in both LUAD and LUSC dirs")
        n_luad = sum(1 for r in records if r["subtype"] == "LUAD")
        n_lusc = sum(1 for r in records if r["subtype"] == "LUSC")
        print(f"Dataset: {len(records)} matched patients (LUAD: {n_luad}, LUSC: {n_lusc})")
        multi = [r for r in records if r["n_slides"] > 1]
        if multi:
            print(f"{len(multi)} patient(s) have multiple slides concatenated "
                  f"into one bag (e.g. {multi[0]['sid']}: {multi[0]['n_slides']} slides)")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        chunks = []
        for p in rec["h5_paths"]:
            with h5py.File(p, "r") as f:
                feats = torch.tensor(f["features"][:], dtype=torch.float32)
            if feats.ndim == 3 and feats.shape[0] == 1:
                feats = feats.squeeze(0)
            elif feats.ndim != 2:
                raise ValueError(f"Unexpected shape {tuple(feats.shape)} in {p}")
            chunks.append(feats)
        features = torch.cat(chunks, dim=0)

        if self.n_tiles_cap is not None and features.shape[0] > self.n_tiles_cap:
            perm = torch.randperm(features.shape[0])[: self.n_tiles_cap]
            features = features[perm]

        label = torch.tensor(rec["label"], dtype=torch.float32)
        clinical = torch.tensor(rec["clinical"], dtype=torch.float32)
        subtype_id = torch.tensor(0 if rec["subtype"] == "LUAD" else 1, dtype=torch.long)
        return features, label, label.clone(), clinical, subtype_id, rec["sid"]


import h5py
print("CPTACExternalDataset defined.")

In [ ]:
# Load config + metadata, build dataset
with open(f"{MODEL_CONFIG_DIR}/model_config.json") as f:
    config = json.load(f)

gene_cols = config["gene_cols"]
clinical_cols = config["clinical_cols"]
raw_gene_cols = [c for c in gene_cols if c.endswith("_fpkm_uq")]
apm_genes = config["APM_GENES"]
tis_genes = config["TIS_GENES"]
N_FOLDS = config["training"]["n_folds"]


df = pd.read_csv(METADATA_CSV)
print(f"Loaded metadata: {len(df)} rows, {df['submitter_id'].nunique()} unique submitter_id")

missing_cols = [c for c in gene_cols + clinical_cols if c not in df.columns]
if missing_cols:
    print(f"\u26a0 CSV is missing expected columns: {missing_cols}")
    print("Check this against model_config.json's gene_cols/clinical_cols before continuing.")

test_ds = CPTACExternalDataset(
    df,
    feature_dirs={"LUAD": LUAD_DIRS, "LUSC": LUSC_DIRS},
    gene_cols=gene_cols,
    clinical_cols=clinical_cols,
    n_tiles_cap=None,  # set e.g. 20000 if hit GPU OOM is hit on multi-slide patients
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

test_sids = [r["sid"] for r in test_ds.records]
test_subtypes = [r["subtype"] for r in test_ds.records]

## Run inference for each fold checkpoint, for each architecture

Same fixed test set for every fold and every architecture (all CPTAC
cases), matching how the TCGA held-out test set was evaluated: for each
architecture in CHECKPOINT_DIRS, N_FOLDS independently-trained models are
loaded and run against one shared test set.

In [ ]:
def run_inference(model, loader, device):
    model.eval()
    all_preds, all_labels, all_sids = [], [], []
    with torch.no_grad():
        for features, label, _, clinical, subtype_id, sid in loader:
            features   = features.squeeze(0).to(device)
            label      = label.squeeze(0).to(device)
            clinical   = clinical.squeeze(0).to(device)
            subtype_id = subtype_id.squeeze(0).to(device)

            pred, _ = model(features, clinical, subtype_id)
            all_preds.append(pred.cpu().numpy())
            all_labels.append(label.cpu().numpy())
            all_sids.append(sid[0])
    return np.array(all_preds), np.array(all_labels), all_sids


def build_model_for_architecture(architecture):
    return FiLMMILModel(
        feat_dim=1536, n_genes=len(gene_cols), use_film=FILM_ENABLED,
        use_panel_head=USE_PANEL_HEAD, aggregator=architecture,
        agg_hidden_dim=AGG_HIDDEN_DIM, graph_k=GRAPH_K, graph_layers=GRAPH_LAYERS,
        transmil_heads=TRANSMIL_HEADS, transmil_layers=TRANSMIL_LAYERS,
        transmil_dim_feedforward=TRANSMIL_DIM_FEEDFORWARD,
    ).to(device)


all_fold_preds_by_arch = {}   # {architecture: {fold: preds}}
ensemble_preds_by_arch = {}   # {architecture: preds}
all_fold_labels = None
sid_order = None

for architecture, ckpt_dir in CHECKPOINT_DIRS.items():
    print(f"\n{'='*60}\nArchitecture: {architecture}\n{'='*60}")
    fold_preds = {}
    for fold in range(N_FOLDS):
        ckpt_path = Path(ckpt_dir) / f"fold{fold}_best_model.pt"
        if not ckpt_path.exists():
            print(f"  Fold {fold}: checkpoint not found at {ckpt_path}, skipping")
            continue

        model = build_model_for_architecture(architecture)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

        preds, labels, sids = run_inference(model, test_loader, device)
        fold_preds[fold] = preds
        all_fold_labels = labels   # identical every fold/architecture (same fixed test set)
        sid_order = sids
        print(f"  Fold {fold}: inference done, preds shape {preds.shape}")

    if not fold_preds:
        print(f"  No checkpoints found for '{architecture}' -- skipping this architecture entirely.")
        continue

    all_fold_preds_by_arch[architecture] = fold_preds
    ensemble_preds_by_arch[architecture] = np.mean(list(fold_preds.values()), axis=0)
    print(f"  -> Ensemble computed from {len(fold_preds)} fold(s)")

ARCHITECTURES = list(ensemble_preds_by_arch.keys())
print(f"\n\u2713 Loaded architectures: {ARCHITECTURES}")
if len(ARCHITECTURES) < 2:
    print("\u26a0 Fewer than 2 architectures loaded -- the cross-architecture "
          "comparison cells later in this notebook need at least 2 to do anything.")

### Save raw predictions

In [ ]:
for architecture in ARCHITECTURES:
    fold_preds = all_fold_preds_by_arch[architecture]
    for fold, preds in fold_preds.items():
        np.savez(
            f"{OUTPUT_DIR}/{architecture}_fold{fold}_cptac_predictions.npz",
            preds=preds, labels=all_fold_labels,
            submitter_id=np.array(sid_order), subtype=np.array(test_subtypes),
            gene_cols=np.array(gene_cols),
        )
    np.savez(
        f"{OUTPUT_DIR}/{architecture}_ensemble_cptac_predictions.npz",
        preds=ensemble_preds_by_arch[architecture], labels=all_fold_labels,
        submitter_id=np.array(sid_order), subtype=np.array(test_subtypes),
        gene_cols=np.array(gene_cols),
    )

print(f"Saved raw predictions for {len(ARCHITECTURES)} architecture(s) to {OUTPUT_DIR}")

## Analysis

### Panel PCC / AUC — per fold and ensemble

Same panel-score definition used throughout (mean of z-scored panel genes), so this is directly comparable to the TCGA held-out test-set numbers.

In [ ]:
from sklearn.metrics import roc_auc_score

def panel_score(arr, gene_cols, panel_genes):
    cols_idx = [gene_cols.index(f"{g}_fpkm_uq") for g in panel_genes if f"{g}_fpkm_uq" in gene_cols]
    return arr[:, cols_idx].mean(axis=1)

def panel_pcc(preds, labels, gene_cols, panel_genes):
    x = panel_score(preds, gene_cols, panel_genes)
    y = panel_score(labels, gene_cols, panel_genes)
    return float(np.corrcoef(x, y)[0, 1])

def panel_auc(preds, labels, gene_cols, panel_genes):
    pred_score = panel_score(preds, gene_cols, panel_genes)
    label_score = panel_score(labels, gene_cols, panel_genes)
    thresh = np.percentile(label_score, 75)
    y_true = (label_score >= thresh).astype(int)
    if y_true.sum() in (0, len(y_true)):
        return float("nan")
    return float(roc_auc_score(y_true, pred_score))

results_summary_by_arch = {}
for architecture in ARCHITECTURES:
    fold_preds = all_fold_preds_by_arch[architecture]
    ensemble_preds = ensemble_preds_by_arch[architecture]
    results_summary = {}
    for name, preds in {**{f"fold{f}": p for f, p in fold_preds.items()}, "ensemble": ensemble_preds}.items():
        results_summary[name] = {}
        for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
            pcc = panel_pcc(preds, all_fold_labels, gene_cols, genes)
            auc = panel_auc(preds, all_fold_labels, gene_cols, genes)
            results_summary[name][panel] = {"PCC": pcc, "AUC": auc}
    results_summary_by_arch[architecture] = results_summary
    print(f"\n{architecture}:")
    for panel in ["APM", "TIS"]:
        e = results_summary["ensemble"][panel]
        print(f"  ensemble {panel}: PCC={e['PCC']:.4f}  AUC={e['AUC']:.4f}")

with open(f"{OUTPUT_DIR}/cptac_results_summary_by_architecture.json", "w") as f:
    json.dump(results_summary_by_arch, f, indent=2)

# Consolidated comparison table (ensemble only), sorted by APM PCC
comparison_rows = []
for architecture in ARCHITECTURES:
    row = {"architecture": architecture}
    for panel in ["APM", "TIS"]:
        e = results_summary_by_arch[architecture]["ensemble"][panel]
        row[f"{panel}_PCC"] = e["PCC"]
        row[f"{panel}_AUC"] = e["AUC"]
    comparison_rows.append(row)
architecture_comparison_df = pd.DataFrame(comparison_rows).sort_values("APM_PCC", ascending=False).reset_index(drop=True)
architecture_comparison_df.to_csv(f"{OUTPUT_DIR}/architecture_comparison_ensemble.csv", index=False)
print("\nEnsemble comparison across architectures:")
print(architecture_comparison_df.to_string(index=False))

### Bootstrap CIs + permutation p-values

Same approach as the TCGA test-set stats — resampling at the **patient** level (since each patient contributes exactly one row here regardless of how many slides went into their prediction).

### Subtype-stratified breakdown (LUAD vs LUSC)

In [ ]:
subtypes_arr = np.array(test_subtypes)
subtype_rows = []
for architecture in ARCHITECTURES:
    ensemble_preds = ensemble_preds_by_arch[architecture]
    for subtype in ["LUAD", "LUSC"]:
        mask = subtypes_arr == subtype
        n = mask.sum()
        for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
            pcc = panel_pcc(ensemble_preds[mask], all_fold_labels[mask], gene_cols, genes)
            subtype_rows.append({"architecture": architecture, "subtype": subtype, "panel": panel, "n": int(n), "PCC": pcc})

subtype_df = pd.DataFrame(subtype_rows)
subtype_df.to_csv(f"{OUTPUT_DIR}/cptac_subtype_breakdown_by_architecture.csv", index=False)
print(subtype_df.to_string(index=False))

# Pivot for a quick eyeball: does the LUAD vs LUSC gap change across architectures?
subtype_pivot = subtype_df.pivot_table(index=["architecture", "panel"], columns="subtype", values="PCC")
subtype_pivot["LUAD_minus_LUSC"] = subtype_pivot["LUAD"] - subtype_pivot["LUSC"]
print("\nLUAD - LUSC PCC gap per architecture/panel:")
print(subtype_pivot.to_string())

### Compare against TCGA held-out test-set numbers

The TCGA test-set PCCs (from the earlier `results.json` / `bootstrap_permutation_stats.json`) are filled in here for comparison.

In [ ]:
N_BOOT = 2000
N_PERM = 2000
rng = np.random.default_rng(0)

def bootstrap_ci(preds, labels, gene_list, n_boot=N_BOOT):
    n = preds.shape[0]
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        vals.append(panel_pcc(preds[idx], labels[idx], gene_cols, gene_list))
    vals = np.array(vals)
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def permutation_pvalue(preds, labels, gene_list, n_perm=N_PERM):
    observed = panel_pcc(preds, labels, gene_cols, gene_list)
    n = preds.shape[0]
    null_vals = np.empty(n_perm)
    for i in range(n_perm):
        perm_idx = rng.permutation(n)
        null_vals[i] = panel_pcc(preds, labels[perm_idx], gene_cols, gene_list)
    p = (np.sum(null_vals >= observed) + 1) / (n_perm + 1)
    return float(observed), float(p)

boot_stats_by_arch = {}
for architecture in ARCHITECTURES:
    fold_preds = all_fold_preds_by_arch[architecture]
    ensemble_preds = ensemble_preds_by_arch[architecture]
    boot_stats = {}
    for name, preds in {**{f"fold{f}": p for f, p in fold_preds.items()}, "ensemble": ensemble_preds}.items():
        boot_stats[name] = {}
        for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
            mean_pcc, lo, hi = bootstrap_ci(preds, all_fold_labels, genes)
            observed, p = permutation_pvalue(preds, all_fold_labels, genes)
            boot_stats[name][panel] = {"PCC": observed, "CI95_low": lo, "CI95_high": hi, "perm_p": p}
    boot_stats_by_arch[architecture] = boot_stats
    print(f"\n{architecture} (ensemble):")
    for panel in ["APM", "TIS"]:
        s = boot_stats["ensemble"][panel]
        print(f"  {panel}: PCC={s['PCC']:.4f} 95%CI=[{s['CI95_low']:.4f},{s['CI95_high']:.4f}] p={s['perm_p']:.4f}")

with open(f"{OUTPUT_DIR}/cptac_bootstrap_stats_by_architecture.json", "w") as f:
    json.dump(boot_stats_by_arch, f, indent=2)

In [ ]:
# Fill these in per architecture from each architecture's TCGA-held-out-test
# results.json (or the aggregator_sweep_summary.json produced by the
# training notebook's sweep) -- e.g. results.json -> ensemble -> APM -> PCC.
TCGA_TEST_PCC = {
    "abmil":    {"APM": None, "TIS": None},
    "clam":     {"APM": None, "TIS": None},
    "transmil": {"APM": None, "TIS": None},
    "graph":    {"APM": None, "TIS": None},
}

print("Internal (TCGA) vs External (CPTAC) ensemble PCC, by architecture:")
tcga_vs_cptac_rows = []
for architecture in ARCHITECTURES:
    for panel in ["APM", "TIS"]:
        cptac_pcc = results_summary_by_arch[architecture]["ensemble"][panel]["PCC"]
        tcga_pcc = TCGA_TEST_PCC.get(architecture, {}).get(panel)
        if tcga_pcc is not None:
            delta = cptac_pcc - tcga_pcc
            print(f"  {architecture:>9s} {panel}: TCGA={tcga_pcc:.4f}  CPTAC={cptac_pcc:.4f}  delta={delta:+.4f}")
        else:
            delta = None
            print(f"  {architecture:>9s} {panel}: CPTAC={cptac_pcc:.4f}  "
                  f"(fill in TCGA_TEST_PCC[{architecture!r}] above to compare)")
        tcga_vs_cptac_rows.append({
            "architecture": architecture, "panel": panel,
            "TCGA_PCC": tcga_pcc, "CPTAC_PCC": cptac_pcc, "delta": delta,
        })

tcga_vs_cptac_df = pd.DataFrame(tcga_vs_cptac_rows)
tcga_vs_cptac_df.to_csv(f"{OUTPUT_DIR}/tcga_vs_cptac_by_architecture.csv", index=False)

### Per-gene PCC for CPTAC (with FDR)

In [ ]:
from scipy.stats import pearsonr

try:
    from statsmodels.stats.multitest import multipletests
    def fdr_correct(pvals):
        return multipletests(pvals, method="fdr_bh")[1]
except ImportError:
    def fdr_correct(pvals):
        pvals = np.asarray(pvals); n = len(pvals)
        order = np.argsort(pvals)
        ranked = pvals[order] * n / (np.arange(n) + 1)
        ranked = np.minimum.accumulate(ranked[::-1])[::-1]
        out = np.empty(n); out[order] = np.clip(ranked, 0, 1)
        return out

gene_symbols = [c.replace("_fpkm_uq", "") for c in gene_cols if c.endswith("_fpkm_uq")]
gene_idx = [gene_cols.index(f"{g}_fpkm_uq") for g in gene_symbols]

gene_pcc_rows = []
gene_pcc_by_arch = {}   # architecture -> {gene: PCC}, reused by the cross-architecture cells below
for architecture in ARCHITECTURES:
    ensemble_preds = ensemble_preds_by_arch[architecture]
    per_gene = {}
    for g, i in zip(gene_symbols, gene_idx):
        r, p = pearsonr(ensemble_preds[:, i], all_fold_labels[:, i])
        panel = "APM" if g in apm_genes else ("TIS" if g in tis_genes else "other")
        gene_pcc_rows.append({"architecture": architecture, "gene": g, "PCC": r, "p": p, "panel": panel})
        per_gene[g] = r
    gene_pcc_by_arch[architecture] = per_gene

gene_df = pd.DataFrame(gene_pcc_rows)
gene_df["q"] = fdr_correct(gene_df["p"].values)
gene_df = gene_df.sort_values(["architecture", "PCC"], ascending=[True, False])
gene_df.to_csv(f"{OUTPUT_DIR}/cptac_gene_level_pcc_by_architecture.csv", index=False)
print(gene_df.to_string(index=False))

### Subtype-level bootstrap CIs

In [ ]:
subtype_ci_rows = []
for architecture in ARCHITECTURES:
    ensemble_preds = ensemble_preds_by_arch[architecture]
    for subtype in ["LUAD", "LUSC"]:
        mask = subtypes_arr == subtype
        for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
            mean_pcc, lo, hi = bootstrap_ci(ensemble_preds[mask], all_fold_labels[mask], genes)
            observed, p = permutation_pvalue(ensemble_preds[mask], all_fold_labels[mask], genes)
            subtype_ci_rows.append({
                "architecture": architecture, "subtype": subtype, "panel": panel, "n": int(mask.sum()),
                "PCC": observed, "CI95_low": lo, "CI95_high": hi, "perm_p": p,
            })

subtype_ci_df = pd.DataFrame(subtype_ci_rows)
subtype_ci_df.to_csv(f"{OUTPUT_DIR}/cptac_subtype_breakdown_with_CI_by_architecture.csv", index=False)
print(subtype_ci_df.to_string(index=False))

## Cross-Architecture Comparison

Every earlier cell above answers "how good is architecture X?" one architecture
at a time. The cells below directly answer the question that actually
matters for the paper: **is an observed PCC gap between two architectures
real, or could it be noise at this sample size?**

A naive way to check this -- look at whether two architectures' independent
95% CIs overlap -- overstates the uncertainty of the *difference*, because
it ignores that both architectures are evaluated on the exact same patients
and are therefore subject to correlated sampling noise. The paired
bootstrap below resamples patients ONCE per iteration and applies that same
resample to both architectures, so the shared noise cancels out and the
resulting CI is a direct, correctly-calibrated answer to "is this gap
robust?" -- including broken down by LUAD vs LUSC, since an architecture
could plausibly be better for one subtype and not the other.

### Paired bootstrap ΔPCC between architecture pairs

`ΔPCC(A, B) = PCC(A) − PCC(B)` computed on the SAME bootstrap resample of
patients for both A and B, repeated `N_BOOT` times. The 95% CI of ΔPCC
directly answers whether the apparent gap is robust: if the CI excludes
zero, the direction of the difference is unlikely to be noise.

In [ ]:
def paired_bootstrap_delta(preds_a, preds_b, labels, gene_list, mask=None, n_boot=N_BOOT):
    """
    Paired bootstrap: the SAME resampled patient indices are applied to
    both architectures' predictions (and to labels) on each iteration, so
    shared sampling noise across architectures on the same patients cancels
    out. This is the correct way to test whether an observed PCC gap
    between two architectures is robust, rather than checking whether
    their independent CIs happen to overlap.
    """
    if mask is not None:
        preds_a, preds_b, labels = preds_a[mask], preds_b[mask], labels[mask]
    n = preds_a.shape[0]
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        pcc_a = panel_pcc(preds_a[idx], labels[idx], gene_cols, gene_list)
        pcc_b = panel_pcc(preds_b[idx], labels[idx], gene_cols, gene_list)
        deltas[i] = pcc_a - pcc_b
    observed = (panel_pcc(preds_a, labels, gene_cols, gene_list)
                - panel_pcc(preds_b, labels, gene_cols, gene_list))
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    # Two-sided bootstrap p-value: proportion of resamples on the "wrong"
    # side of zero, doubled (capped at 1.0).
    p_boot = min(2 * min(np.mean(deltas <= 0), np.mean(deltas >= 0)), 1.0)
    return {
        "delta_PCC": float(observed), "CI95_low": float(lo), "CI95_high": float(hi),
        "p_boot": float(p_boot), "robust": bool(lo > 0 or hi < 0),
    }


delta_rows = []
subtype_groups = {"ALL": None, "LUAD": subtypes_arr == "LUAD", "LUSC": subtypes_arr == "LUSC"}

for arch_a, arch_b in itertools.combinations(ARCHITECTURES, 2):
    preds_a = ensemble_preds_by_arch[arch_a]
    preds_b = ensemble_preds_by_arch[arch_b]
    for panel, genes in [("APM", apm_genes), ("TIS", tis_genes)]:
        for group_name, mask in subtype_groups.items():
            stats = paired_bootstrap_delta(preds_a, preds_b, all_fold_labels, genes, mask=mask)
            delta_rows.append({
                "arch_A": arch_a, "arch_B": arch_b, "panel": panel, "subtype_group": group_name,
                "n": int(mask.sum()) if mask is not None else len(all_fold_labels),
                **stats,
            })

delta_df = pd.DataFrame(delta_rows)
delta_df.to_csv(f"{OUTPUT_DIR}/architecture_pairwise_delta_pcc_bootstrap.csv", index=False)

print(f"Pairwise ΔPCC = PCC(arch_A) - PCC(arch_B), paired bootstrap 95% CI (n_boot={N_BOOT}):")
print("('robust' = True means the 95% CI excludes zero, i.e. the direction "
      "of the difference is unlikely to be noise)\n")
print(delta_df.to_string(index=False))

robust_diffs = delta_df[delta_df["robust"]]
if len(robust_diffs):
    print(f"\n{len(robust_diffs)} robust pairwise difference(s) found (CI excludes 0):")
    print(robust_diffs[["arch_A", "arch_B", "panel", "subtype_group",
                         "delta_PCC", "CI95_low", "CI95_high"]].to_string(index=False))
else:
    print("\nNo pairwise architecture difference had a 95% CI excluding zero -- "
          "apparent PCC gaps between architectures are not distinguishable "
          "from noise at this sample size.")

### How similar are the architectures' predictions? (r_gene between architectures)

Two complementary diagnostics:
1. **Prediction agreement per gene** — for each gene, correlate the two
   architectures' predicted values across patients. High agreement means
   the architectures are outputting near-identical predictions for that
   gene (redundant); low agreement means they're picking up different
   signal (useful for ensembling, and a sign that any apparent PCC edge
   might be gene-specific rather than a general improvement).
2. **Per-gene performance correlation** — correlate the two architectures'
   per-gene PCC vectors (across all genes). High correlation means they're
   good/bad at the same genes (structurally similar); low correlation means
   they have complementary strengths.

In [ ]:
# (1) Per-gene prediction agreement between architecture pairs
agreement_rows = []
for arch_a, arch_b in itertools.combinations(ARCHITECTURES, 2):
    preds_a = ensemble_preds_by_arch[arch_a]
    preds_b = ensemble_preds_by_arch[arch_b]
    for g, i in zip(gene_symbols, gene_idx):
        r, _ = pearsonr(preds_a[:, i], preds_b[:, i])
        panel = "APM" if g in apm_genes else ("TIS" if g in tis_genes else "other")
        agreement_rows.append({"arch_A": arch_a, "arch_B": arch_b, "gene": g, "panel": panel, "r_gene": r})

agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(f"{OUTPUT_DIR}/architecture_pairwise_prediction_agreement_per_gene.csv", index=False)

print("Prediction agreement (r_gene) between architecture pairs, summarised across genes:")
print(agreement_df.groupby(["arch_A", "arch_B"])["r_gene"].agg(["mean", "median", "min", "max"]).round(3).to_string())

# (2) Per-gene PERFORMANCE correlation between architecture pairs
perf_corr_rows = []
for arch_a, arch_b in itertools.combinations(ARCHITECTURES, 2):
    pcc_a = np.array([gene_pcc_by_arch[arch_a][g] for g in gene_symbols])
    pcc_b = np.array([gene_pcc_by_arch[arch_b][g] for g in gene_symbols])
    r, p = pearsonr(pcc_a, pcc_b)
    perf_corr_rows.append({"arch_A": arch_a, "arch_B": arch_b, "r_per_gene_PCC": r, "p": p})

perf_corr_df = pd.DataFrame(perf_corr_rows)
perf_corr_df.to_csv(f"{OUTPUT_DIR}/architecture_pairwise_per_gene_PCC_correlation.csv", index=False)
print("\nCorrelation between architectures' PER-GENE PCC vectors "
      "(high = good/bad at the same genes; low = complementary strengths):")
print(perf_corr_df.to_string(index=False))

# Heatmap of mean prediction agreement across genes -- quick visual summary
if len(ARCHITECTURES) >= 2:
    agreement_matrix = pd.DataFrame(index=ARCHITECTURES, columns=ARCHITECTURES, dtype=float)
    for a in ARCHITECTURES:
        for b in ARCHITECTURES:
            if a == b:
                agreement_matrix.loc[a, b] = 1.0
            else:
                key = agreement_df[((agreement_df.arch_A == a) & (agreement_df.arch_B == b)) |
                                    ((agreement_df.arch_A == b) & (agreement_df.arch_B == a))]
                agreement_matrix.loc[a, b] = key["r_gene"].mean()

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(agreement_matrix.values.astype(float), vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(ARCHITECTURES))); ax.set_xticklabels(ARCHITECTURES, rotation=45, ha="right")
    ax.set_yticks(range(len(ARCHITECTURES))); ax.set_yticklabels(ARCHITECTURES)
    for i in range(len(ARCHITECTURES)):
        for j in range(len(ARCHITECTURES)):
            ax.text(j, i, f"{agreement_matrix.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title("Mean prediction agreement (r_gene)\nacross all genes")
    fig.colorbar(im, ax=ax, shrink=0.8, label="mean r_gene")
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/architecture_prediction_agreement_heatmap.png", dpi=150)
    plt.show()

### Patient-level manifest

In [ ]:
manifest = pd.DataFrame({
    "submitter_id": test_sids,
    "subtype": test_subtypes,
    "n_slides": [r["n_slides"] for r in test_ds.records],
})
manifest.to_csv(f"{OUTPUT_DIR}/cptac_patient_manifest.csv", index=False)
print(f"{len(manifest)} patients in final CPTAC test set "
      f"({(manifest.subtype=='LUAD').sum()} LUAD, {(manifest.subtype=='LUSC').sum()} LUSC)")
print(f"Multi-slide patients: {(manifest.n_slides > 1).sum()}")

### Self-document the run

In [ ]:
shutil.copy(f"{MODEL_CONFIG_DIR}/model_config.json", f"{OUTPUT_DIR}/model_config_used.json")

run_metadata = {
    "architectures_loaded": ARCHITECTURES,
    "n_folds_loaded_per_architecture": {a: len(all_fold_preds_by_arch[a]) for a in ARCHITECTURES},
    "n_test_patients": len(test_ds),
    "tcga_vs_cptac_ensemble_by_architecture": {
        a: {
            panel: {
                "TCGA": TCGA_TEST_PCC.get(a, {}).get(panel),
                "CPTAC": results_summary_by_arch[a]["ensemble"][panel]["PCC"],
            }
            for panel in ["APM", "TIS"]
        }
        for a in ARCHITECTURES
    },
    "output_files": {
        "per_architecture_results": "cptac_results_summary_by_architecture.json",
        "ensemble_comparison_table": "architecture_comparison_ensemble.csv",
        "subtype_breakdown": "cptac_subtype_breakdown_by_architecture.csv",
        "bootstrap_stats": "cptac_bootstrap_stats_by_architecture.json",
        "subtype_bootstrap_stats": "cptac_subtype_breakdown_with_CI_by_architecture.csv",
        "per_gene_pcc": "cptac_gene_level_pcc_by_architecture.csv",
        "pairwise_delta_pcc_bootstrap": "architecture_pairwise_delta_pcc_bootstrap.csv",
        "pairwise_prediction_agreement": "architecture_pairwise_prediction_agreement_per_gene.csv",
        "pairwise_per_gene_pcc_correlation": "architecture_pairwise_per_gene_PCC_correlation.csv",
    },
}
with open(f"{OUTPUT_DIR}/run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2)
print("Saved model_config_used.json and run_metadata.json")